In [1]:
# 2.1 Import Library dan Konfigurasi Path
import pandas as pd
import numpy as np
import os

# Konfigurasi path
DATA_PATH = '../../../stemming/data_preprocessing_final.csv'
SLA_LEXICON_PATH = '../../../stemming/outputs/SLA/sla_lexicon_adapted.csv' # Memuat leksikon yang sudah dibuat
POLITIK_PATH = '../../../kamus/inset_vader_political_modified.csv'
OUTPUT_DIR = '../../../stemming/outputs/SLA'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("[INFO] Library dan konfigurasi path berhasil dimuat.")

[INFO] Library dan konfigurasi path berhasil dimuat.


In [2]:
# 2.2 Load Data Preprocessing Final
df = pd.read_csv(DATA_PATH)

print(f"\nData preprocessing berhasil dimuat: {len(df)} tweet")
print(f"Kolom: {df.columns.tolist()}")
df.head()


Data preprocessing berhasil dimuat: 13192 tweet
Kolom: ['no', 'timestamp', 'teks', 'teks_processed']


,no,timestamp,teks,teks_processed
0,1,2016-12-30T06:37:56.000Z,ADIL loh utk yg punya kebijakan publik negara ...,ADIL loh untuk yang punya kebijakan publik neg...
1,2,2016-12-30T06:30:36.000Z,Tertibkan Media Online DPR Pemerintah Jangan S...,tertib media online DPR pemerintah jangan spor...
2,3,2016-12-30T04:48:35.000Z,harus dievaluasi lg kebijakan bebas visa truta...,harus evaluasi lagi kebijakan bebas visa utama...
3,4,2016-12-30T04:21:40.000Z,jangan ngambang aturan logis apa undang undang,jangan ngambang pengaturan logis apa undang un...
4,5,2016-12-30T02:36:13.000Z,Kebebasan bersuara berpendapat memang dijamin ...,bebas suara dapat memang jamin UU tetapi bebas...


In [3]:
# 2.3 Load Leksikon (InSet SLA + Politik SLA)

# 1. Load Leksikon SLA Adaptasi
df_inset_sla = pd.read_csv(SLA_LEXICON_PATH)
df_inset_sla['kata'] = df_inset_sla['kata'].astype(str).str.strip().str.lower()

# 2. Load Leksikon Politik
df_politik_sla = pd.read_csv(POLITIK_PATH)
df_politik_sla['kata'] = df_politik_sla['kata'].astype(str).str.strip().str.lower()

# 3. Gabungkan Kedua DataFrame
df_combined = pd.concat([df_inset_sla, df_politik_sla]).drop_duplicates(subset='kata', keep='last')

# 4. Buat Dictionary untuk Matching
sla_dict = dict(zip(df_combined['kata'], df_combined['mean']))

print(f"[INFO] Leksikon Berhasil Digabung.")
print(f"       - InSet SLA   : {len(df_inset_sla)} entri")
print(f"       - Politik     : {len(df_politik_sla)} entri")
print(f"       - Total Gabung: {len(sla_dict)} entri unik")

[INFO] Leksikon Berhasil Digabung.
       - InSet SLA   : 9074 entri
       - Politik     : 49 entri
       - Total Gabung: 9101 entri unik


In [4]:
# 2.4 Definisi Kategori Kata Fungsi
NEGASI_DAN_MODAL = {
    'tidak', 'bukan', 'jangan', 'belum', 'sangat', 'harus', 'wajib',
    'akan', 'sudah', 'sedang', 'telah', 'boleh', 'bisa'
}
KATA_HUBUNG_PREPOSISI = {
    'dan', 'atau', 'tetapi', 'karena', 'jika', 'di', 'ke', 'dari',
    'pada', 'untuk', 'dengan', 'oleh', 'hingga', 'sejak'
}
PRONOMINA_DEMONSTRATIVA = {
    'saya', 'aku', 'dia', 'kami', 'kamu', 'anda', 'ini', 'itu', 'yang'
}
PARTIKEL_KATA_TANYA = {
    'pun', 'sih', 'ya', 'lah', 'kah', 'apa', 'siapa', 'bagaimana'
}

ALL_FUNCTION_WORDS = (
    NEGASI_DAN_MODAL
    | KATA_HUBUNG_PREPOSISI
    | PRONOMINA_DEMONSTRATIVA
    | PARTIKEL_KATA_TANYA
)

In [5]:
# 2.5 Diagnostik Kata Fungsi dalam Leksikon SLA (Tanpa Perlakuan)
df_found_sla = df_combined[df_combined['kata'].isin(ALL_FUNCTION_WORDS)].copy().sort_values('kata')

print(f"[DIAGNOSTIK] Total kata fungsi yang TERDAFTAR di leksikon Gabungan: {len(df_found_sla)}")
print("\n[KONTEKS] Kata-kata ini akan dianggap bermuatan sentimen (masuk ke matched_words)")
print("karena tidak ada perlakuan 'ignore' maupun 'hapus'.")

if not df_found_sla.empty:
    print("\nContoh kata fungsi yang akan dihitung skornya (Mean SLA):")
    print(df_found_sla[['kata', 'mean']].head(10).to_string(index=False))
else:
    print("Tidak ditemukan kata fungsi di leksikon.")

[DIAGNOSTIK] Total kata fungsi yang TERDAFTAR di leksikon Gabungan: 21

[KONTEKS] Kata-kata ini akan dianggap bermuatan sentimen (masuk ke matched_words)
karena tidak ada perlakuan 'ignore' maupun 'hapus'.

Contoh kata fungsi yang akan dihitung skornya (Mean SLA):
  kata  mean
   aku   1.6
  anda  -3.2
   apa  -2.4
 boleh   1.6
 bukan  -2.4
  dari  -2.4
   dia  -2.4
 harus  -4.0
   itu  -1.6
jangan  -2.4


In [6]:
# 2.6 Konfigurasi Dictionary SLA (Tanpa Perlakuan Fungsi)
sla_dict_full = dict(zip(df_combined['kata'].astype(str).str.lower(), df_combined['mean'].astype(float)))
ignore_set_empty = set()

print(f"[KONFIGURASI] Dictionary SLA Gabungan siap: {len(sla_dict_full)} entri.")
print(f"[INFO] Perlakuan Fungsi 3: Tidak ada kata fungsi yang dinetralkan atau dihapus.")

[KONFIGURASI] Dictionary SLA Gabungan siap: 9101 entri.
[INFO] Perlakuan Fungsi 3: Tidak ada kata fungsi yang dinetralkan atau dihapus.


In [7]:
# 2.6 Fungsi Tokenisasi
def tokenize(text):
    if not isinstance(text, str):
        return []
    return text.split()

df['tokens'] = df['teks_processed'].apply(tokenize)
print(f"\n[INFO] Tokenisasi selesai. Total token: {df['tokens'].str.len().sum():,}")


[INFO] Tokenisasi selesai. Total token: 235,560


In [8]:
# 2.7 Fungsi Lexicon Matching (Tanpa Perlakuan)
# Jika kata ada di kamus -> Matched. Jika tidak -> Unmatched.
# Tidak ada filter untuk kata fungsi.

def match_lexicon_no_ignore(tokens, lexicon):
    matched_words = []
    matched_function_words = [] # Kolom tambahan untuk analisis dampak kata fungsi
    unmatched_words = []
    
    for token in tokens:
        token_lower = token.lower()
        if token_lower in lexicon:
            matched_words.append(token)
            # Cek apakah kata yang matched ini sebenarnya adalah kata fungsi
            if token_lower in ALL_FUNCTION_WORDS:
                matched_function_words.append(token)
        else:
            unmatched_words.append(token)
           
    return matched_words, matched_function_words, unmatched_words

print("[INFO] Fungsi matching tanpa ignore siap.")

[INFO] Fungsi matching tanpa ignore siap.


In [9]:
# 2.8 Penerapan Lexicon Matching
print("\n[PROSES] Menjalankan lexicon matching (Tanpa Perlakuan Fungsi)...")

df[['matched_words', 'matched_function_words', 'unmatched_words']] = df['tokens'].apply(
    lambda x: pd.Series(match_lexicon_no_ignore(x, sla_dict_full))
)

print("[INFO] Lexicon matching selesai.")


[PROSES] Menjalankan lexicon matching (Tanpa Perlakuan Fungsi)...
[INFO] Lexicon matching selesai.


In [10]:
# 2.9 Perhitungan Statistik
total_words = df['tokens'].str.len().sum()
total_matched = df['matched_words'].str.len().sum()
total_matched_func = df['matched_function_words'].str.len().sum()
total_unmatched = df['unmatched_words'].str.len().sum()

# Kata fungsi yang dianggap bermuatan sentimen
func_as_sentiment = total_matched_func

print("\n[STATISTIK] Hasil Lexicon Matching (SLA + Tanpa Perlakuan):")
print(f"Total kata               : {total_words:,}")
print(f"Matched di SLA           : {total_matched:,} ({(total_matched/total_words)*100:.2f}%)")
print(f"   -> Termasuk Kata Fungsi: {func_as_sentiment:,} ({(func_as_sentiment/total_words)*100:.2f}%)")
print(f"Unmatched                : {total_unmatched:,} ({(total_unmatched/total_words)*100:.2f}%)")
print("\n[ANALISIS] Jumlah 'Matched Termasuk Kata Fungsi'.")
print("menunjukkan potensi bias sentimen karena kata fungsi dihitung sebagai kata bermuatan.")


[STATISTIK] Hasil Lexicon Matching (SLA + Tanpa Perlakuan):
Total kata               : 235,560
Matched di SLA           : 123,218 (52.31%)
   -> Termasuk Kata Fungsi: 12,652 (5.37%)
Unmatched                : 112,342 (47.69%)

[ANALISIS] Jumlah 'Matched Termasuk Kata Fungsi'.
menunjukkan potensi bias sentimen karena kata fungsi dihitung sebagai kata bermuatan.


In [11]:
# 2.10 Preview Hasil Matching
print("\n[PREVIEW] 3 Tweet Pertama:")
for i in range(3):
    print(f"\nTweet {i+1}: {df['teks_processed'].iloc[i][:80]}...")
    print(f"  Matched (Total)      : {df['matched_words'].iloc[i][:5]}")
    print(f"  Matched (Kata Fungsi): {df['matched_function_words'].iloc[i][:5]}")
    print(f"  Unmatched            : {df['unmatched_words'].iloc[i][:5]}")


[PREVIEW] 3 Tweet Pertama:

Tweet 1: ADIL loh untuk yang punya kebijakan publik negara ingat yang ini ! !...
  Matched (Total)      : ['ADIL', 'yang', 'punya', 'kebijakan', 'ingat']
  Matched (Kata Fungsi): ['yang', 'yang']
  Unmatched            : ['loh', 'untuk', 'publik', 'negara', 'ini']

Tweet 2: tertib media online DPR pemerintah jangan sporadis apalagi selektif hanya kepada...
  Matched (Total)      : ['tertib', 'DPR', 'pemerintah', 'jangan', 'sporadis']
  Matched (Kata Fungsi): ['jangan', 'yang']
  Unmatched            : ['media', 'online', 'apalagi', 'kepada', 'media']

Tweet 3: harus evaluasi lagi kebijakan bebas visa utama untuk negara tiongkok pak ! ! bah...
  Matched (Total)      : ['harus', 'lagi', 'kebijakan', 'bebas', 'bahaya']
  Matched (Kata Fungsi): ['harus']
  Unmatched            : ['evaluasi', 'visa', 'utama', 'untuk', 'negara']


In [12]:
# 2.11 Simpan Output
output_path = os.path.join(OUTPUT_DIR, 'lexicon_matching_tanpa_ignore.csv')
df.to_csv(output_path, index=False)
print(f"\n[OUTPUT] Data berhasil disimpan ke: {output_path}")


[OUTPUT] Data berhasil disimpan ke: ../../../stemming/outputs/SLA\lexicon_matching_tanpa_ignore.csv
